# Transformer Fine-Tuning Basics

This notebook builds the missing bridge between **transformer foundations** and **PEFT**. We will pretrain a tiny transformer encoder on unlabeled support text, then fine-tune it on an intent-classification task using two strategies:

- **Frozen backbone + task head** (linear probing)
- **Full fine-tuning** (update the whole transformer)

By the end, you will understand:

1. What a **task head** is and why it is usually tiny
2. The standard **supervised fine-tuning flow**
3. When freezing the backbone is enough
4. Why small datasets create **overfitting risk**
5. When you should move from full fine-tuning to **PEFT** methods like LoRA


## 1. Setup

We keep all notebook hyperparameters in one place so the experiments stay easy to adjust and reproduce.


In [ ]:
import copy
import math
import random
from collections import Counter

import lightning as L
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.loggers import CSVLogger
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
import importlib.util
from pathlib import Path

repo_root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
utils_path = repo_root / 'src/aiml_notebooks/utils.py'
spec = importlib.util.spec_from_file_location('aiml_notebooks_utils', utils_path)
utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(utils)
count_parameters = utils.count_parameters
get_device = utils.get_device
set_seed = utils.set_seed

CONFIG = {
    # Reproducibility
    'seed': 42,

    # Synthetic corpus sizes
    'unlabeled_examples_per_intent': 180,
    'labeled_train_per_intent': 24,
    'labeled_val_per_intent': 12,
    'labeled_test_per_intent': 12,

    # Tokenization
    'max_length': 24,
    'min_token_frequency': 1,
    'mlm_probability': 0.15,

    # Data loading
    'batch_size': 32,
    'num_workers': 0,

    # Backbone architecture
    'd_model': 64,
    'num_heads': 4,
    'num_layers': 2,
    'ff_dim': 128,
    'dropout': 0.1,

    # Pretraining
    'mlm_learning_rate': 3e-3,
    'mlm_max_epochs': 5,
    'mlm_early_stop_patience': 2,

    # Fine-tuning
    'ft_learning_rate': 2e-3,
    'ft_max_epochs': 12,
    'ft_early_stop_patience': 3,
    'classifier_dropout': 0.2,

    # Reporting
    'memory_bytes_per_param': 16,
}

pd.set_option('display.max_colwidth', 120)
sns.set_theme(style='whitegrid')
print('Configuration loaded.')


### Random Seed and Device

This notebook uses a tiny transformer, so CPU execution is fast enough and avoids MPS transformer edge cases on macOS.


In [ ]:
set_seed(CONFIG['seed'])
L.seed_everything(CONFIG['seed'], workers=True)
device = get_device(prefer_cpu=True)
print(f'Chosen device: {device}')


## 2. Build a Tiny Support Domain

Real fine-tuning starts with a pretrained model and a downstream task. To keep this notebook self-contained, we simulate that setup in a small customer-support domain.


In [ ]:
INTENTS = {
    'billing': {
        'subjects': ['invoice', 'refund', 'charge', 'payment', 'subscription'],
        'issues': ['looks incorrect', 'was charged twice', 'did not go through', 'needs a refund', 'renewed unexpectedly'],
        'requests': ['fix the billing error', 'explain the statement', 'issue a refund', 'update the payment method'],
    },
    'shipping': {
        'subjects': ['package', 'shipment', 'delivery', 'tracking', 'order'],
        'issues': ['is delayed', 'has not moved', 'arrived damaged', 'was sent to the wrong address', 'is still pending'],
        'requests': ['share the latest tracking update', 'replace the order', 'speed up the shipment', 'confirm the address'],
    },
    'account': {
        'subjects': ['account', 'password', 'login', 'profile', 'verification'],
        'issues': ['is locked', 'keeps failing', 'needs to be reset', 'is missing information', 'requires verification'],
        'requests': ['restore access', 'reset the password', 'unlock the profile', 'verify the account'],
    },
    'cancellation': {
        'subjects': ['plan', 'membership', 'trial', 'subscription', 'renewal'],
        'issues': ['should be cancelled', 'renewed unexpectedly', 'is no longer needed', 'must stop before the next cycle', 'needs to end today'],
        'requests': ['cancel immediately', 'confirm the cancellation', 'stop the renewal', 'close the membership'],
    },
}

CHANNELS = ['email', 'chat', 'phone', 'support portal']
TONES = ['frustrated', 'calm', 'urgent', 'confused']
TIMES = ['today', 'this morning', 'last night', 'after the weekend', 'during checkout']
CUSTOMERS = ['new customer', 'long-time customer', 'enterprise admin', 'student subscriber']

def generate_support_text(intent, rng):
    template = INTENTS[intent]
    subject = rng.choice(template['subjects'])
    issue = rng.choice(template['issues'])
    request = rng.choice(template['requests'])
    channel = rng.choice(CHANNELS)
    tone = rng.choice(TONES)
    time_phrase = rng.choice(TIMES)
    customer = rng.choice(CUSTOMERS)

    sentence_a = f"A {tone} {customer} contacted support through {channel}."
    sentence_b = f"Their {subject} {issue} {time_phrase}."
    sentence_c = f"They want the team to {request}."
    sentence_d = f"The ticket should be routed to the {intent} queue."
    return ' '.join([sentence_a, sentence_b, sentence_c, sentence_d])

rng = random.Random(CONFIG['seed'])


### Generate Unlabeled and Labeled Text

The unlabeled corpus stands in for **pretraining data**. The labeled set stands in for a downstream fine-tuning dataset.


In [ ]:
unlabeled_texts = []
for intent in INTENTS:
    for _ in range(CONFIG['unlabeled_examples_per_intent']):
        unlabeled_texts.append(generate_support_text(intent, rng))

labeled_rows = []
per_intent_total = (
    CONFIG['labeled_train_per_intent']
    + CONFIG['labeled_val_per_intent']
    + CONFIG['labeled_test_per_intent']
)
for intent in INTENTS:
    for _ in range(per_intent_total):
        labeled_rows.append({'text': generate_support_text(intent, rng), 'label': intent})

labeled_df = pd.DataFrame(labeled_rows)
print(f'Unlabeled documents: {len(unlabeled_texts)}')
print(f'Labeled examples: {len(labeled_df)}')
print('\nSample labeled examples:')
display(labeled_df.sample(3, random_state=CONFIG['seed']))


### Visualize Class Balance

A balanced dataset makes it easier to isolate the fine-tuning question instead of confounding it with class imbalance.


In [ ]:
label_counts = labeled_df['label'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(label_counts.index, label_counts.values, color=['#4C78A8', '#F58518', '#54A24B', '#E45756'])
ax.set_title('Synthetic downstream dataset by intent')
ax.set_ylabel('Number of examples')
ax.set_xlabel('Intent')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 3. Tokenization and Vocabulary

A transformer needs token IDs, attention masks, and special tokens. We will use a lightweight word-level tokenizer because the goal is understanding fine-tuning, not tokenization mechanics.


In [ ]:
SPECIAL_TOKENS = ['[PAD]', '[UNK]', '[CLS]', '[MASK]']
PAD_TOKEN, UNK_TOKEN, CLS_TOKEN, MASK_TOKEN = SPECIAL_TOKENS

def simple_tokenize(text):
    return text.lower().replace('.', ' .').split()

all_training_texts = unlabeled_texts + labeled_df['text'].tolist()
token_counter = Counter()
for text in all_training_texts:
    token_counter.update(simple_tokenize(text))

vocab = SPECIAL_TOKENS + sorted(
    token for token, count in token_counter.items()
    if count >= CONFIG['min_token_frequency'] and token not in SPECIAL_TOKENS
)

token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for token, idx in token_to_id.items()}
pad_idx = token_to_id[PAD_TOKEN]
unk_idx = token_to_id[UNK_TOKEN]
cls_idx = token_to_id[CLS_TOKEN]
mask_idx = token_to_id[MASK_TOKEN]

print(f'Vocabulary size: {len(vocab)}')
print('First 16 tokens:', vocab[:16])


### Encode a Sentence

We prepend a **[CLS]** token because the classifier head will read that position as a summary of the sequence.


In [ ]:
def encode_text(text, max_length):
    tokens = [CLS_TOKEN] + simple_tokenize(text)
    token_ids = [token_to_id.get(token, unk_idx) for token in tokens][:max_length]
    attention_mask = [1] * len(token_ids)

    if len(token_ids) < max_length:
        padding = [pad_idx] * (max_length - len(token_ids))
        token_ids = token_ids + padding
        attention_mask = attention_mask + [0] * len(padding)

    return torch.tensor(token_ids, dtype=torch.long), torch.tensor(attention_mask, dtype=torch.long)


def decode_ids(token_ids):
    return ' '.join(id_to_token[int(idx)] for idx in token_ids if int(idx) != pad_idx)

sample_ids, sample_mask = encode_text(labeled_df.iloc[0]['text'], CONFIG['max_length'])
print('Decoded sample:')
print(decode_ids(sample_ids))
print('Attention mask:', sample_mask.tolist())


### Check Sequence Lengths

This helps us verify that `max_length` is large enough without wasting too much padding.


In [ ]:
lengths = [min(len(simple_tokenize(text)) + 1, CONFIG['max_length']) for text in all_training_texts]
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(lengths, bins=np.arange(0, CONFIG['max_length'] + 2) - 0.5, color='#72B7B2', edgecolor='white')
ax.set_title('Encoded sequence lengths (including [CLS])')
ax.set_xlabel('Sequence length')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()
print(f'Max observed length before truncation: {max(lengths)}')


## 4. Prepare the Downstream Splits

We create train/validation/test splits per intent so the evaluation stays stable.


In [ ]:
label_to_idx = {label: idx for idx, label in enumerate(sorted(INTENTS.keys()))}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

split_frames = []
for label, group in labeled_df.groupby('label', sort=True):
    group = group.sample(frac=1.0, random_state=CONFIG['seed']).reset_index(drop=True)
    train_end = CONFIG['labeled_train_per_intent']
    val_end = train_end + CONFIG['labeled_val_per_intent']

    split_frames.append(group.iloc[:train_end].assign(split='train'))
    split_frames.append(group.iloc[train_end:val_end].assign(split='val'))
    split_frames.append(group.iloc[val_end:].assign(split='test'))

split_df = pd.concat(split_frames, ignore_index=True)
print(split_df.groupby(['split', 'label']).size().unstack(fill_value=0))


### Inspect the Fine-Tuning Dataflow

A standard supervised fine-tuning setup has three ingredients:

- tokenized inputs
- a pretrained backbone
- a small task-specific head that maps backbone features to labels


In [ ]:
class MaskedLanguageModelingDataset(Dataset):
    def __init__(self, texts, max_length):
        self.texts = texts
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        input_ids, attention_mask = encode_text(self.texts[idx], self.max_length)
        return input_ids, attention_mask


class IntentClassificationDataset(Dataset):
    def __init__(self, frame, max_length):
        self.frame = frame.reset_index(drop=True)
        self.max_length = max_length

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        input_ids, attention_mask = encode_text(row['text'], self.max_length)
        label = label_to_idx[row['label']]
        return input_ids, attention_mask, torch.tensor(label, dtype=torch.long)


def create_mlm_batch(batch):
    input_ids = torch.stack([item[0] for item in batch])
    attention_mask = torch.stack([item[1] for item in batch])

    probability_matrix = torch.full(input_ids.shape, CONFIG['mlm_probability'])
    special_mask = (input_ids == pad_idx) | (input_ids == cls_idx)
    probability_matrix.masked_fill_(special_mask, value=0.0)

    masked_indices = torch.bernoulli(probability_matrix).bool()
    labels = input_ids.clone()
    labels[~masked_indices] = -100

    replace_prob = torch.rand(input_ids.shape)
    masked_inputs = input_ids.clone()
    masked_inputs[masked_indices & (replace_prob < 0.8)] = mask_idx

    random_tokens = torch.randint(low=len(SPECIAL_TOKENS), high=len(vocab), size=input_ids.shape)
    masked_inputs[masked_indices & (replace_prob >= 0.8) & (replace_prob < 0.9)] = random_tokens[
        masked_indices & (replace_prob >= 0.8) & (replace_prob < 0.9)
    ]

    return masked_inputs, attention_mask, labels


### Create DataLoaders

We will use one set of loaders for masked-LM pretraining and another for downstream classification.


In [ ]:
pretrain_dataset = MaskedLanguageModelingDataset(unlabeled_texts, CONFIG['max_length'])
train_dataset = IntentClassificationDataset(split_df[split_df['split'] == 'train'], CONFIG['max_length'])
val_dataset = IntentClassificationDataset(split_df[split_df['split'] == 'val'], CONFIG['max_length'])
test_dataset = IntentClassificationDataset(split_df[split_df['split'] == 'test'], CONFIG['max_length'])

pretrain_loader = DataLoader(
    pretrain_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    collate_fn=create_mlm_batch,
)
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)

masked_ids, masked_attention, masked_labels = next(iter(pretrain_loader))
print('MLM batch shapes:', masked_ids.shape, masked_attention.shape, masked_labels.shape)
print('Classification batch shapes:', [tensor.shape for tensor in next(iter(train_loader))])


## 5. Backbone and Task Heads

The backbone learns general-purpose representations. The head is the tiny task-specific layer you swap when the downstream task changes.


In [ ]:
class TinyTransformerBackbone(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, ff_dim, max_length, dropout):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_length, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask):
        positions = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x = self.dropout(x)
        key_padding_mask = attention_mask == 0
        encoded = self.encoder(x, src_key_padding_mask=key_padding_mask)
        encoded = self.norm(encoded)
        cls_representation = encoded[:, 0]
        return cls_representation, encoded


class MaskedLanguageModel(nn.Module):
    def __init__(self, backbone, vocab_size):
        super().__init__()
        self.backbone = backbone
        self.mlm_head = nn.Linear(CONFIG['d_model'], vocab_size)

    def forward(self, input_ids, attention_mask):
        _, token_features = self.backbone(input_ids, attention_mask)
        return self.mlm_head(token_features)


class IntentClassifier(nn.Module):
    def __init__(self, backbone, num_classes, dropout):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(CONFIG['d_model'], num_classes),
        )

    def forward(self, input_ids, attention_mask):
        cls_representation, _ = self.backbone(input_ids, attention_mask)
        return self.classifier(cls_representation)


### Sanity-Check the Backbone

Before training, we verify that the shapes match the modeling assumptions.


In [ ]:
backbone = TinyTransformerBackbone(
    vocab_size=len(vocab),
    d_model=CONFIG['d_model'],
    num_heads=CONFIG['num_heads'],
    num_layers=CONFIG['num_layers'],
    ff_dim=CONFIG['ff_dim'],
    max_length=CONFIG['max_length'],
    dropout=CONFIG['dropout'],
)
mlm_model = MaskedLanguageModel(backbone, vocab_size=len(vocab))

cls_representation, token_features = backbone(masked_ids[:4], masked_attention[:4])
mlm_logits = mlm_model(masked_ids[:4], masked_attention[:4])
print('CLS representation:', cls_representation.shape)
print('Token features:', token_features.shape)
print('MLM logits:', mlm_logits.shape)


### Visualize the MLM Corruption Step

Masked language modeling teaches the backbone to predict hidden tokens from context. That creates a useful starting point for downstream fine-tuning.


In [ ]:
example_original_ids, _ = pretrain_dataset[0]
example_masked_ids, _, example_labels = create_mlm_batch([pretrain_dataset[0]])
masked_positions = (example_labels[0] != -100).nonzero(as_tuple=False).flatten().tolist()

print('Original :', decode_ids(example_original_ids))
print('Masked   :', decode_ids(example_masked_ids[0]))
print('Masked positions:', masked_positions)


## 6. Lightning Module for Pretraining

We use PyTorch Lightning so the training loops stay compact and the logged metrics are easy to compare later.


In [ ]:
class MaskedLMModule(L.LightningModule):
    def __init__(self, model, learning_rate):
        super().__init__()
        self.model = model
        self.learning_rate = learning_rate

    def forward(self, input_ids, attention_mask):
        return self.model(input_ids, attention_mask)

    def _shared_step(self, batch, stage):
        input_ids, attention_mask, labels = batch
        logits = self(input_ids, attention_mask)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100)

        valid_mask = labels != -100
        if valid_mask.any():
            predictions = logits.argmax(dim=-1)
            accuracy = (predictions[valid_mask] == labels[valid_mask]).float().mean()
        else:
            accuracy = torch.tensor(0.0, device=self.device)

        self.log(f'{stage}_loss', loss, on_step=False, on_epoch=True, prog_bar=(stage == 'val'))
        self.log(f'{stage}_acc', accuracy, on_step=False, on_epoch=True, prog_bar=(stage == 'val'))
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, 'train')

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, 'val')

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.learning_rate)


def build_trainer(run_name, max_epochs, patience):
    logger = CSVLogger('logs', name='transformer_fine_tuning_basics', version=run_name)
    early_stop = EarlyStopping(monitor='val_loss', patience=patience, mode='min')
    trainer = L.Trainer(
        max_epochs=max_epochs,
        accelerator='cpu',
        devices=1,
        logger=logger,
        callbacks=[early_stop],
        deterministic=True,
        enable_progress_bar=False,
        enable_checkpointing=False,
        log_every_n_steps=1,
        num_sanity_val_steps=0,
    )
    return trainer, logger


def extract_epoch_metrics(log_dir):
    metrics = pd.read_csv(f'{log_dir}/metrics.csv')
    grouped = metrics.groupby('epoch').max(numeric_only=True).reset_index()
    return grouped


### Pretrain the Backbone with MLM

This is our stand-in for generic transformer pretraining. The notebook keeps it small, but the workflow matches the real idea: learn a reusable backbone before touching the downstream labels.


In [ ]:
pretrain_train_size = int(0.85 * len(pretrain_dataset))
pretrain_val_size = len(pretrain_dataset) - pretrain_train_size
pretrain_train_dataset, pretrain_val_dataset = torch.utils.data.random_split(
    pretrain_dataset,
    [pretrain_train_size, pretrain_val_size],
    generator=torch.Generator().manual_seed(CONFIG['seed']),
)

pretrain_train_loader = DataLoader(
    pretrain_train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=0,
    collate_fn=create_mlm_batch,
)
pretrain_val_loader = DataLoader(
    pretrain_val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=0,
    collate_fn=create_mlm_batch,
)

mlm_module = MaskedLMModule(mlm_model, learning_rate=CONFIG['mlm_learning_rate'])
pretrain_trainer, pretrain_logger = build_trainer(
    run_name='mlm_pretraining',
    max_epochs=CONFIG['mlm_max_epochs'],
    patience=CONFIG['mlm_early_stop_patience'],
)
pretrain_trainer.fit(mlm_module, pretrain_train_loader, pretrain_val_loader)
print('Pretraining log directory:', pretrain_logger.log_dir)


### Plot the MLM Learning Curves

If the backbone is learning useful domain structure, masked-token accuracy should rise while validation loss falls.


In [ ]:
pretrain_metrics = extract_epoch_metrics(pretrain_logger.log_dir)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(pretrain_metrics['epoch'], pretrain_metrics['train_loss'], marker='o', label='Train')
axes[0].plot(pretrain_metrics['epoch'], pretrain_metrics['val_loss'], marker='s', label='Validation')
axes[0].set_title('MLM loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(pretrain_metrics['epoch'], pretrain_metrics['train_acc'], marker='o', label='Train')
axes[1].plot(pretrain_metrics['epoch'], pretrain_metrics['val_acc'], marker='s', label='Validation')
axes[1].set_title('Masked-token accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

print(pretrain_metrics[['epoch', 'train_loss', 'val_loss', 'train_acc', 'val_acc']].round(3))


## 7. Define the Fine-Tuning Objective

Now we attach a classification head to the pretrained backbone. This is the standard fine-tuning recipe:

1. keep the pretrained backbone
2. add a task head
3. choose which parameters stay frozen
4. optimize on labeled examples


In [ ]:
class IntentClassifierModule(L.LightningModule):
    def __init__(self, model, learning_rate, freeze_backbone=False):
        super().__init__()
        self.model = model
        self.learning_rate = learning_rate
        self.freeze_backbone = freeze_backbone
        self.loss_fn = nn.CrossEntropyLoss()

        if freeze_backbone:
            for parameter in self.model.backbone.parameters():
                parameter.requires_grad = False

    def forward(self, input_ids, attention_mask):
        return self.model(input_ids, attention_mask)

    def _shared_step(self, batch, stage):
        input_ids, attention_mask, labels = batch
        logits = self(input_ids, attention_mask)
        loss = self.loss_fn(logits, labels)
        predictions = logits.argmax(dim=-1)
        accuracy = (predictions == labels).float().mean()
        self.log(f'{stage}_loss', loss, on_step=False, on_epoch=True, prog_bar=(stage == 'val'))
        self.log(f'{stage}_acc', accuracy, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, 'train')

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, 'val')

    def test_step(self, batch, batch_idx):
        self._shared_step(batch, 'test')

    def configure_optimizers(self):
        params = [p for p in self.parameters() if p.requires_grad]
        return torch.optim.AdamW(params, lr=self.learning_rate)


### Create Two Starting Points from the Same Backbone

Both strategies start from identical pretrained weights. The only difference is whether the backbone stays frozen.


In [ ]:
pretrained_backbone_state = copy.deepcopy(mlm_module.model.backbone.state_dict())

frozen_backbone = TinyTransformerBackbone(
    vocab_size=len(vocab),
    d_model=CONFIG['d_model'],
    num_heads=CONFIG['num_heads'],
    num_layers=CONFIG['num_layers'],
    ff_dim=CONFIG['ff_dim'],
    max_length=CONFIG['max_length'],
    dropout=CONFIG['dropout'],
)
full_ft_backbone = TinyTransformerBackbone(
    vocab_size=len(vocab),
    d_model=CONFIG['d_model'],
    num_heads=CONFIG['num_heads'],
    num_layers=CONFIG['num_layers'],
    ff_dim=CONFIG['ff_dim'],
    max_length=CONFIG['max_length'],
    dropout=CONFIG['dropout'],
)

frozen_backbone.load_state_dict(pretrained_backbone_state)
full_ft_backbone.load_state_dict(pretrained_backbone_state)

frozen_classifier = IntentClassifier(frozen_backbone, num_classes=len(label_to_idx), dropout=CONFIG['classifier_dropout'])
full_ft_classifier = IntentClassifier(full_ft_backbone, num_classes=len(label_to_idx), dropout=CONFIG['classifier_dropout'])

frozen_module = IntentClassifierModule(frozen_classifier, CONFIG['ft_learning_rate'], freeze_backbone=True)
full_ft_module = IntentClassifierModule(full_ft_classifier, CONFIG['ft_learning_rate'], freeze_backbone=False)


### Compare Trainable Parameter Counts

The task head is tiny compared with the backbone. This is why frozen-backbone training is cheap.


In [ ]:
parameter_summary = pd.DataFrame([
    {
        'strategy': 'Frozen backbone + head',
        'total_params': count_parameters(frozen_module),
        'trainable_params': count_parameters(frozen_module, trainable_only=True),
    },
    {
        'strategy': 'Full fine-tuning',
        'total_params': count_parameters(full_ft_module),
        'trainable_params': count_parameters(full_ft_module, trainable_only=True),
    },
])
parameter_summary['trainable_ratio'] = parameter_summary['trainable_params'] / parameter_summary['total_params']
display(parameter_summary.style.format({'total_params': '{:,.0f}', 'trainable_params': '{:,.0f}', 'trainable_ratio': '{:.2%}'}))


## 8. Fine-Tune Strategy 1: Frozen Backbone + Task Head

This is often called a **linear probe** or **feature extraction** setup. It is fast and stable, but it can only reuse features the backbone already knows how to produce.


In [ ]:
frozen_trainer, frozen_logger = build_trainer(
    run_name='frozen_backbone',
    max_epochs=CONFIG['ft_max_epochs'],
    patience=CONFIG['ft_early_stop_patience'],
)
frozen_trainer.fit(frozen_module, train_loader, val_loader)
print('Frozen-backbone log directory:', frozen_logger.log_dir)


### Plot the Frozen-Backbone Curves

Watch how quickly the small task head adapts when the representation is already useful.


In [ ]:
frozen_metrics = extract_epoch_metrics(frozen_logger.log_dir)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(frozen_metrics['epoch'], frozen_metrics['train_loss'], marker='o', label='Train')
axes[0].plot(frozen_metrics['epoch'], frozen_metrics['val_loss'], marker='s', label='Validation')
axes[0].set_title('Frozen backbone loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(frozen_metrics['epoch'], frozen_metrics['train_acc'], marker='o', label='Train')
axes[1].plot(frozen_metrics['epoch'], frozen_metrics['val_acc'], marker='s', label='Validation')
axes[1].set_title('Frozen backbone accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()


## 9. Fine-Tune Strategy 2: Update the Whole Transformer

Full fine-tuning can adapt the representation itself, not just the head. That usually gives more capacity, but it also increases memory use and overfitting risk.


In [ ]:
full_ft_trainer, full_ft_logger = build_trainer(
    run_name='full_fine_tune',
    max_epochs=CONFIG['ft_max_epochs'],
    patience=CONFIG['ft_early_stop_patience'],
)
full_ft_trainer.fit(full_ft_module, train_loader, val_loader)
print('Full fine-tuning log directory:', full_ft_logger.log_dir)


### Plot the Full Fine-Tuning Curves

Here the backbone and head both move. That adds flexibility, but it also makes the optimization problem larger.


In [ ]:
full_ft_metrics = extract_epoch_metrics(full_ft_logger.log_dir)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(full_ft_metrics['epoch'], full_ft_metrics['train_loss'], marker='o', label='Train')
axes[0].plot(full_ft_metrics['epoch'], full_ft_metrics['val_loss'], marker='s', label='Validation')
axes[0].set_title('Full fine-tuning loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(full_ft_metrics['epoch'], full_ft_metrics['train_acc'], marker='o', label='Train')
axes[1].plot(full_ft_metrics['epoch'], full_ft_metrics['val_acc'], marker='s', label='Validation')
axes[1].set_title('Full fine-tuning accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()


## 10. Compare the Two Strategies

The interesting question is not just “which wins,” but **why**. Frozen-backbone training is cheap and hard to destabilize. Full fine-tuning is more expressive but easier to overfit.


In [ ]:
comparison = pd.DataFrame({
    'epoch': frozen_metrics['epoch'],
    'frozen_val_acc': frozen_metrics['val_acc'],
    'full_val_acc': full_ft_metrics['val_acc'],
    'frozen_val_loss': frozen_metrics['val_loss'],
    'full_val_loss': full_ft_metrics['val_loss'],
})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(comparison['epoch'], comparison['frozen_val_acc'], marker='o', label='Frozen backbone')
axes[0].plot(comparison['epoch'], comparison['full_val_acc'], marker='s', label='Full fine-tune')
axes[0].set_title('Validation accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(comparison['epoch'], comparison['frozen_val_loss'], marker='o', label='Frozen backbone')
axes[1].plot(comparison['epoch'], comparison['full_val_loss'], marker='s', label='Full fine-tune')
axes[1].set_title('Validation loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()


### Evaluate on the Held-Out Test Set

Validation curves help us tune. The test set tells us whether the final choice generalizes.


In [ ]:
def collect_predictions(module, loader):
    module.eval()
    predictions = []
    labels = []
    with torch.no_grad():
        for input_ids, attention_mask, batch_labels in loader:
            logits = module(input_ids, attention_mask)
            predictions.extend(logits.argmax(dim=-1).cpu().tolist())
            labels.extend(batch_labels.cpu().tolist())
    return np.array(labels), np.array(predictions)

frozen_true, frozen_pred = collect_predictions(frozen_module, test_loader)
full_true, full_pred = collect_predictions(full_ft_module, test_loader)

test_results = pd.DataFrame([
    {'strategy': 'Frozen backbone + head', 'test_accuracy': accuracy_score(frozen_true, frozen_pred)},
    {'strategy': 'Full fine-tuning', 'test_accuracy': accuracy_score(full_true, full_pred)},
])
display(test_results.style.format({'test_accuracy': '{:.2%}'}))


### Visualize the Confusion Matrices

A confusion matrix shows where the model still mixes up downstream intents, not just how often it is right overall.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, true_labels, pred_labels, title in [
    (axes[0], frozen_true, frozen_pred, 'Frozen backbone + head'),
    (axes[1], full_true, full_pred, 'Full fine-tuning'),
]:
    cm = confusion_matrix(true_labels, pred_labels, labels=list(range(len(label_to_idx))))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=list(label_to_idx.keys()), yticklabels=list(label_to_idx.keys()))
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
plt.tight_layout()
plt.show()


### Read the Classification Report

Precision, recall, and F1 make the tradeoffs easier to interpret than raw accuracy alone.


In [ ]:
print('Frozen backbone + head')
print(classification_report(frozen_true, frozen_pred, target_names=list(label_to_idx.keys()), digits=3))
print('Full fine-tuning')
print(classification_report(full_true, full_pred, target_names=list(label_to_idx.keys()), digits=3))


## 11. Overfitting Risk

A classic fine-tuning pitfall is assuming that “more trainable parameters” always helps. On small datasets, it often widens the gap between training performance and validation performance.


In [ ]:
def summarize_training_gap(metrics, name):
    final_row = metrics.iloc[-1]
    return {
        'strategy': name,
        'train_acc': final_row['train_acc'],
        'val_acc': final_row['val_acc'],
        'accuracy_gap': final_row['train_acc'] - final_row['val_acc'],
        'train_loss': final_row['train_loss'],
        'val_loss': final_row['val_loss'],
        'loss_gap': final_row['val_loss'] - final_row['train_loss'],
    }

gap_summary = pd.DataFrame([
    summarize_training_gap(frozen_metrics, 'Frozen backbone + head'),
    summarize_training_gap(full_ft_metrics, 'Full fine-tuning'),
])
display(gap_summary.style.format({
    'train_acc': '{:.2%}',
    'val_acc': '{:.2%}',
    'accuracy_gap': '{:.2%}',
    'train_loss': '{:.3f}',
    'val_loss': '{:.3f}',
    'loss_gap': '{:.3f}',
}))


### Why the Gap Matters

A large train/validation gap usually means the model adapted too specifically to the labeled set. This is why fine-tuning workflows always need:

- a validation split
- early stopping or checkpoint selection
- a task-appropriate metric, not just training loss


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(gap_summary['strategy'], gap_summary['accuracy_gap'], color=['#4C78A8', '#E45756'])
ax.set_title('Train vs validation accuracy gap')
ax.set_ylabel('Gap (train acc - val acc)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 12. The Supervised Fine-Tuning Flow

The mechanics of fine-tuning are usually straightforward. The important engineering choices happen around data quality, evaluation, and parameter budgets.


In [ ]:
ft_flow = pd.DataFrame([
    ['1. Start from a pretrained checkpoint', 'Reuse general language features instead of training from scratch'],
    ['2. Add a task head', 'Map backbone features to logits, spans, tags, or scores'],
    ['3. Choose what to freeze', 'Trade off stability and cost against adaptability'],
    ['4. Train on labeled data', 'Optimize the downstream objective with validation monitoring'],
    ['5. Evaluate and stress-test', 'Check for overfitting, class failures, and calibration issues'],
    ['6. Escalate to PEFT if needed', 'Use LoRA or related methods when full updates become too expensive'],
], columns=['Step', 'Why it matters'])
display(ft_flow)


## 13. When Does PEFT Become Necessary?

For small encoders, full fine-tuning is often feasible. For larger transformer backbones, optimizer states and gradients dominate memory. That is where **PEFT** becomes attractive.


In [ ]:
def estimate_full_ft_memory_gb(parameter_count_millions):
    total_params = parameter_count_millions * 1_000_000
    total_bytes = total_params * CONFIG['memory_bytes_per_param']
    return total_bytes / 1e9

memory_table = pd.DataFrame([
    ['Tiny notebook encoder', count_parameters(full_ft_module) / 1_000_000],
    ['BERT-base', 110],
    ['RoBERTa-large', 355],
    ['7B LLM', 7000],
], columns=['model', 'parameters_m'])
memory_table['approx_full_ft_memory_gb'] = memory_table['parameters_m'].apply(estimate_full_ft_memory_gb)
memory_table['interpretation'] = [
    'Easy to full fine-tune',
    'Usually feasible on a single modern GPU',
    'Expensive but still manageable on high-memory hardware',
    'Usually needs PEFT, sharding, or quantization',
]
display(memory_table.style.format({'parameters_m': '{:.2f}', 'approx_full_ft_memory_gb': '{:.1f} GB'}))


### Decision Rule of Thumb

You do **not** switch to PEFT because it is trendy. You switch when the full-update path is too expensive, too slow, or too unstable relative to the gain.


In [ ]:
print('Use a frozen backbone when:')
print('  - The labeled dataset is small and the pretrained features are already strong')
print('  - You want a cheap baseline before touching the backbone')
print('\nUse full fine-tuning when:')
print('  - The downstream task is meaningfully different from the pretraining objective')
print('  - The backbone is small enough that full updates are affordable')
print('\nEscalate to PEFT when:')
print('  - Full fine-tuning no longer fits in memory')
print('  - You need many task-specific adapters for one shared backbone')
print('  - You want most of the adaptation benefit without updating every parameter')


## 14. Key Takeaways

This notebook filled the bridge between building transformers and using PEFT methods responsibly.


In [ ]:
print('Key takeaways:')
print('1. Fine-tuning = pretrained backbone + task head + labeled objective.')
print('2. The task head is usually tiny; the backbone holds most of the capacity.')
print('3. Frozen-backbone training is cheap and stable, but only reuses existing features.')
print('4. Full fine-tuning can adapt the representation, but it raises memory use and overfitting risk.')
print('5. Validation curves and held-out metrics are mandatory because small fine-tuning sets are easy to overfit.')
print('6. PEFT becomes attractive when full parameter updates stop being operationally reasonable.')
